# Replay diversity comparison — Colab GPU

Compares the existing V4 continued run (300 distinct replay images repeated three times) with 900 distinct original training images, each once per epoch. Both use 10,800 new-font images and 900 replay presentations per epoch, original epoch-13 initialization, eight epochs, Adam 1e-4 and the same development sets. New run: `replay900_v1`. This is development only; the previous final set was consumed and no new independent final set is defined.

Use a GPU runtime. ZIP/JPEG reads are CPU/I/O; the existing preprocessor, CNN, BiLSTM and CTC run on CUDA. Staging may take time. Checkpoint saves occur after each completed epoch. If interrupted, inspect `last.pt`; the current training script does **not** resume automatically and will refuse to overwrite the run.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
from pathlib import Path
import hashlib, json, os, sys, zipfile, shutil, torch
THESIS=Path('/content/drive/MyDrive/4 Senior/Thesis')
PACKAGE=THESIS/'Manchu-CRNN-Replay-v1'
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print('GPU:',torch.cuda.get_device_name(0),'PyTorch:',torch.__version__)
meta=json.loads((PACKAGE/'MANIFEST.json').read_text())
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
    return h.hexdigest()
assert sha(PACKAGE/'payload.zip')==meta['payload_sha256']
workspace=Path('/content/manchu_replay_v1'); workspace.mkdir(exist_ok=True)
with zipfile.ZipFile(PACKAGE/'payload.zip') as z: z.extractall(workspace)
ROOT=workspace/'research'
print('Package verified; local workspace:',ROOT)


In [ ]:
# Copy the source archive into temporary Colab storage (about 15.7 GB); reuse if present.
source=THESIS/'tyotakuki/New.zip'; archive=workspace/'New.zip'
assert source.stat().st_size==15656444350
if not archive.exists() or archive.stat().st_size != source.stat().st_size:
    assert shutil.disk_usage(workspace).free > source.stat().st_size + 2*1024**3, 'Insufficient Colab disk space'
    temporary=workspace/'New.zip.partial'
    with source.open('rb') as src,temporary.open('wb') as dst:
        shutil.copyfileobj(src,dst,length=16*1024*1024)
    assert temporary.stat().st_size==source.stat().st_size
    temporary.replace(archive)
print('Archive ready:',archive)
sys.path.insert(0,str(PACKAGE))
from prepare import prepare
prepare(PACKAGE,ROOT,archive)


In [ ]:
# This cell starts the full comparison training run.
sys.path.insert(0,str(ROOT/'experiments'))
sys.path.insert(0,str(ROOT/'colab_60k_gpu'))
import train_replay900 as experiment
experiment.ROOT=ROOT
experiment.DATA=ROOT/'data/cross_style_v5'
experiment.RUN=PACKAGE/'runs/replay900_v1'
experiment.BASE=ROOT/'baseline.pt'
assert experiment.sha(experiment.BASE)==experiment.BASE_SHA
experiment.train('cuda')


In [ ]:
# Compare development metrics against the already completed V4 baseline.
result=json.loads((PACKAGE/'runs/replay900_v1/results.json').read_text())
for name in ('mixed_development','original_development'):
    print(name,result['adapted'][name]['overall'])
print('Compare against V4 continued epoch 5: mixed 71.5%, original 97.6%.')
print('No final test was run.')
